In [1]:
try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
    print("TensorFlow: OK")
except Exception as e:
    print("TensorFlow ERROR:", repr(e))

try:
    import shap
    print("SHAP:", shap.__version__)
    print("SHAP: OK")
except Exception as e:
    print("SHAP ERROR:", repr(e))

try:
    import sklearn
    print("scikit-learn:", sklearn.__version__)
    print("scikit-learn: OK")
except Exception as e:
    print("scikit-learn ERROR:", repr(e))

TensorFlow: 2.21.0
TensorFlow: OK
SHAP: 0.52.0
SHAP: OK
scikit-learn: 1.9.1
scikit-learn: OK


# Part 3 – Machine Learning and AI

## Task 3 – Deep Learning with Explainability

### Objective

This task develops a neural network for traffic-demand prediction using the engineered traffic dataset created in Part 2.

A feed-forward neural network is used to predict `traffic_volume` from time, weather and holiday-related features. The model is evaluated using Mean Absolute Error (MAE) and R-squared (R²), allowing its performance to be compared with the supervised regression models developed in Task 1.

Model training behaviour is examined using training and validation loss.

For explainability, SHAP is applied to a comparable Random Forest regression model trained on the same prediction problem. This approach follows the assignment guidance allowing explainability to be applied to a comparable tree-based model when it provides a clearer and more computationally practical explanation of the prediction problem.

The explainability analysis is used to identify which features contribute most strongly to traffic-volume predictions.


In [2]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.propagate = False

if logger.handlers:
    logger.handlers.clear()

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(name)s - %(message)s"
)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

file_handler = logging.FileHandler(
    "task3_deep_learning.log",
    mode="w"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

logger.info(
    "Task 3 deep learning and explainability workflow started."
)

print("TensorFlow:", tf.__version__)
print("SHAP:", shap.__version__)

2026-10-07 21:53:16,202 - INFO - __main__ - Task 3 deep learning and explainability workflow started.


TensorFlow: 2.21.0
SHAP: 0.52.0


## 3.3 Load the Engineered Traffic Dataset

The engineered traffic dataset produced in Part 2 is reused as the input for the deep learning workflow.

Using the same processed data foundation as Task 1 maintains consistency across the supervised and deep learning experiments.

In [3]:
DATA_PATH = Path(
    "../part2_python/engineered_traffic_data.csv"
)

try:
    df = pd.read_csv(
        DATA_PATH,
        parse_dates=["date_time"]
    )

    logger.info(
        "Engineered traffic dataset loaded successfully: "
        "%d rows, %d columns.",
        df.shape[0],
        df.shape[1]
    )

except FileNotFoundError:
    logger.error(
        "Dataset not found: %s",
        DATA_PATH
    )
    raise

except (pd.errors.ParserError, ValueError) as error:
    logger.error(
        "Unable to load dataset: %s",
        error
    )
    raise

print("Dataset shape:", df.shape)

2026-10-07 21:53:41,865 - INFO - __main__ - Engineered traffic dataset loaded successfully: 48187 rows, 30 columns.


Dataset shape: (48187, 30)


## 3.4 Prepare Features for Traffic-Demand Prediction

The neural network uses time, weather and holiday-related predictors while `traffic_volume` is used as the regression target.

Additional Part 3 features are created for holiday status and cyclical day-of-week representation.

Categorical weather conditions are represented using the one-hot encoded weather variables already created in Part 2.

The predictor variables are standardised before neural-network training because neural networks generally train more effectively when numerical inputs are on comparable scales.

In [4]:
# ------------------------------------------------------------
# Additional required features
# ------------------------------------------------------------

holiday_text = (
    df["holiday"]
    .fillna("None")
    .astype(str)
    .str.strip()
    .str.lower()
)

df["is_holiday"] = (
    ~holiday_text.isin(
        ["none", "nan", ""]
    )
).astype(int)

df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)


# ------------------------------------------------------------
# Weather one-hot features
# ------------------------------------------------------------

weather_features = [
    column
    for column in df.columns
    if column.startswith("weather_")
    and column not in [
        "weather_main",
        "weather_description"
    ]
]


# ------------------------------------------------------------
# Predictor set
# ------------------------------------------------------------

base_features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "is_holiday",
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "has_rain",
    "has_snow"
]

feature_columns = []

for column in base_features + weather_features:
    if (
        column in df.columns
        and column not in feature_columns
    ):
        feature_columns.append(column)


X = df[feature_columns].copy()
y = df["traffic_volume"].copy()


if X.isnull().any().any():
    logger.error(
        "Missing values detected in neural-network predictors."
    )
    raise ValueError(
        "Neural-network predictors contain missing values."
    )


logger.info(
    "Neural-network feature set prepared with %d predictors.",
    len(feature_columns)
)

print("Number of predictors:", len(feature_columns))
print("Target:", y.name)
print("Predictor shape:", X.shape)

2026-10-07 21:53:53,197 - INFO - __main__ - Neural-network feature set prepared with 25 predictors.


Number of predictors: 25
Target: traffic_volume
Predictor shape: (48187, 25)


## 3.5 Create Training, Validation and Test Sets

The data is divided into training, validation and test sets.

The training set is used to estimate the neural-network parameters. The validation set is used during training to monitor generalisation and support early stopping. The test set is held back for final evaluation.

A fixed random state is used for reproducibility.


In [5]:
# First split: 80% development, 20% test
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Second split:
# 80% of development for training,
# 20% of development for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.20,
    random_state=42
)


# Standardisation
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_val_scaled = scaler.transform(
    X_val
)

X_test_scaled = scaler.transform(
    X_test
)


logger.info(
    "Deep-learning datasets prepared: "
    "%d train, %d validation, %d test rows.",
    len(X_train),
    len(X_val),
    len(X_test)
)


print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

2026-10-07 21:54:06,897 - INFO - __main__ - Deep-learning datasets prepared: 30839 train, 7710 validation, 9638 test rows.


Training: (30839, 25)
Validation: (7710, 25)
Test: (9638, 25)


## 3.6 Build the Neural Network

A feed-forward neural network is developed to predict traffic volume.

The network contains two hidden dense layers with ReLU activation and a single linear output neuron for regression.

The model is trained using the Adam optimiser and Mean Squared Error (MSE) loss. Mean Absolute Error (MAE) is also monitored during training.

Early stopping is used to reduce unnecessary training and limit overfitting. The model stops when validation loss no longer improves and restores the weights from the best-performing epoch.

In [6]:
# ------------------------------------------------------------
# Task 3.6 - Build neural network
# ------------------------------------------------------------

nn_model = keras.Sequential(
    [
        layers.Input(
            shape=(X_train_scaled.shape[1],)
        ),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dense(
            32,
            activation="relu"
        ),

        layers.Dense(1)
    ],
    name="traffic_demand_neural_network"
)


nn_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


logger.info(
    "Neural network created with input dimension %d.",
    X_train_scaled.shape[1]
)


nn_model.summary()

2026-10-07 21:54:11,183 - INFO - __main__ - Neural network created with input dimension 25.


Model: "traffic_demand_neural_network"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 64)                  │           1,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,777 (14.75 KB)

 Trainable params: 3,777 (14.75 KB)

 Non-trainable params: 0 (0.00 B)

## 3.7 Train the Neural Network

The neural network is trained using the standardised training data and evaluated against the validation set after each epoch.

Early stopping monitors validation loss. If validation performance does not improve for five consecutive epochs, training stops and the best model weights are restored.

A maximum of 50 epochs is specified, although early stopping may terminate training sooner.

In [7]:
# ------------------------------------------------------------
# Task 3.7 - Train neural network
# ------------------------------------------------------------

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


history = nn_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(
        X_val_scaled,
        y_val
    ),
    epochs=50,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)


logger.info(
    "Neural-network training completed after %d epochs.",
    len(history.history["loss"])
)


print(
    "Epochs completed:",
    len(history.history["loss"])
)

Epoch 1/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 10743622.0000 - mae: 2680.8677 - val_loss: 3534570.2500 - val_mae: 1447.9543
Epoch 2/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1749789.8750 - mae: 1011.5836 - val_loss: 1248043.1250 - val_mae: 867.6284
Epoch 3/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1161830.3750 - mae: 832.4583 - val_loss: 1115886.8750 - val_mae: 806.4440
Epoch 4/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1070359.3750 - mae: 785.7723 - val_loss: 1054752.2500 - val_mae: 772.5320
Epoch 5/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1015746.4375 - mae: 757.5384 - val_loss: 1009924.9375 - val_mae: 748.6933
Epoch 6/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 972179.3125 - mae: 735.8142 - val_loss: 969130.6250 - val_mae: 728.3610
Epoch 7/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 928549.0625 - mae: 714.6201 - val_loss: 923765.6250 - val_mae: 706.7313
Epoch 8/50
482/482 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 87

2026-10-07 21:56:00,438 - INFO - __main__ - Neural-network training completed after 50 epochs.


Epochs completed: 50


## 3.8 Neural Network Evaluation

The trained neural network is evaluated using the held-out test dataset.

Performance is measured using:

- **Mean Absolute Error (MAE)** – the average absolute difference between predicted and actual traffic volume.
- **R-squared (R²)** – the proportion of variation in traffic volume explained by the model.

These metrics are consistent with the regression evaluation used in Task 1 and allow the neural network to be compared directly with the Linear Regression and Random Forest regression models.

In [8]:
# ------------------------------------------------------------
# Task 3.8 - Neural-network evaluation
# ------------------------------------------------------------

nn_predictions = (
    nn_model.predict(
        X_test_scaled,
        verbose=0
    )
    .reshape(-1)
)


nn_mae = mean_absolute_error(
    y_test,
    nn_predictions
)

nn_r2 = r2_score(
    y_test,
    nn_predictions
)


logger.info(
    "Neural network evaluated: MAE=%.4f, R2=%.4f.",
    nn_mae,
    nn_r2
)


print(
    f"Neural Network MAE: {nn_mae:.4f}"
)

print(
    f"Neural Network R-squared: {nn_r2:.4f}"
)

2026-10-07 21:56:08,006 - INFO - __main__ - Neural network evaluated: MAE=297.8235, R2=0.9395.


Neural Network MAE: 297.8235
Neural Network R-squared: 0.9395


## 3.9 Training and Validation Loss

Training and validation loss are examined across epochs to assess model convergence and identify possible overfitting.

A large or increasing gap between training and validation loss would suggest that the model is fitting the training data more closely than unseen validation data.

In [9]:
# ------------------------------------------------------------
# Task 3.9 - Training and validation history
# ------------------------------------------------------------

training_loss = history.history["loss"]
validation_loss = history.history["val_loss"]

best_epoch = int(
    np.argmin(validation_loss) + 1
)

best_val_loss = float(
    np.min(validation_loss)
)

final_training_loss = float(
    training_loss[-1]
)

final_validation_loss = float(
    validation_loss[-1]
)


logger.info(
    "Learning history evaluated: best validation epoch=%d.",
    best_epoch
)


print("Epochs completed:", len(training_loss))
print("Best validation epoch:", best_epoch)
print(
    "Best validation loss:",
    round(best_val_loss, 2)
)

print(
    "Final training loss:",
    round(final_training_loss, 2)
)

print(
    "Final validation loss:",
    round(final_validation_loss, 2)
)

2026-10-07 21:56:12,666 - INFO - __main__ - Learning history evaluated: best validation epoch=50.


Epochs completed: 50
Best validation epoch: 50
Best validation loss: 268576.19
Final training loss: 257410.5
Final validation loss: 268576.19


## 3.10 Explainability with SHAP

SHAP is used to explain the traffic-volume prediction problem.

Rather than applying SHAP directly to the neural network, a comparable Random Forest Regressor is trained using the same predictors and prediction target.

This approach is consistent with the assignment guidance, which permits explainability to be applied to a comparable tree-based model when the selected deep learning model is more difficult or computationally expensive to explain directly.

SHAP values quantify how strongly each feature contributes to model predictions. Mean absolute SHAP values are used to rank the overall importance of the predictors.

Because the local Matplotlib installation has a font-rendering issue, SHAP importance is reported numerically rather than through a SHAP plot. This does not affect the SHAP calculations themselves.

In [10]:
# ------------------------------------------------------------
# Task 3.10 - Comparable Random Forest model for SHAP
# ------------------------------------------------------------

shap_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

shap_rf_model.fit(
    X_train,
    y_train
)

shap_rf_predictions = shap_rf_model.predict(
    X_test
)

shap_rf_mae = mean_absolute_error(
    y_test,
    shap_rf_predictions
)

shap_rf_r2 = r2_score(
    y_test,
    shap_rf_predictions
)


logger.info(
    "Comparable Random Forest trained for SHAP: "
    "MAE=%.4f, R2=%.4f.",
    shap_rf_mae,
    shap_rf_r2
)


print(
    f"SHAP Random Forest MAE: {shap_rf_mae:.4f}"
)

print(
    f"SHAP Random Forest R-squared: {shap_rf_r2:.4f}"
)

2026-10-07 21:56:27,675 - INFO - __main__ - Comparable Random Forest trained for SHAP: MAE=275.0783, R2=0.9436.


SHAP Random Forest MAE: 275.0783
SHAP Random Forest R-squared: 0.9436


# 3.11 SHAP Feature Importance

SHAP values are calculated for a reproducible sample of 100 test observations to provide a computationally efficient global explanation of the comparable Random Forest model.

For each feature, the mean absolute SHAP value is calculated. A larger mean absolute SHAP value indicates that the feature has a greater average influence on traffic-volume predictions.

The resulting feature-importance ranking is used to interpret the major drivers of predicted traffic demand.

In [11]:
# ------------------------------------------------------------
# Task 3.11 - SHAP analysis
# ------------------------------------------------------------

shap_sample = X_test.sample(
    n=100,
    random_state=42
)

logger.info(
    "Starting SHAP analysis using %d observations.",
    len(shap_sample)
)

explainer = shap.TreeExplainer(
    shap_rf_model
)

shap_values = explainer.shap_values(
    shap_sample,
    check_additivity=False
)

mean_abs_shap = np.abs(
    shap_values
).mean(axis=0
)

shap_importance = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Mean_Absolute_SHAP": mean_abs_shap
    }
).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

logger.info(
    "SHAP analysis completed successfully."
)

print(
    "Top 10 features by mean absolute SHAP value:"
)

print(
    shap_importance
    .head(10)
    .round(4)
    .to_string(index=False)
)

2026-10-07 21:56:34,266 - INFO - __main__ - Starting SHAP analysis using 100 observations.
2026-10-07 21:59:07,152 - INFO - __main__ - SHAP analysis completed successfully.


Top 10 features by mean absolute SHAP value:
       Feature  Mean_Absolute_SHAP
      hour_cos           1444.8046
          hour            444.4674
      hour_sin            188.3298
       day_sin            147.4430
   day_of_week            144.3160
          temp            106.9698
    is_weekend            101.1651
       day_cos             65.6146
    clouds_all             36.3087
weather_Clouds             12.4848


## 3.12 SHAP Explainability Findings

SHAP was applied to a comparable Random Forest Regressor trained on the same traffic-volume prediction problem as the neural network.

The analysis used a reproducible sample of 100 test observations. Mean absolute SHAP values were used to measure the average magnitude of each feature's contribution to the model predictions.

The most influential features were:

| Rank | Feature | Mean Absolute SHAP |
|---:|---|---:|
| 1 | `hour_cos` | 1444.8046 |
| 2 | `hour` | 444.4674 |
| 3 | `hour_sin` | 188.3298 |
| 4 | `day_sin` | 147.4430 |
| 5 | `day_of_week` | 144.3160 |
| 6 | `temp` | 106.9698 |
| 7 | `is_weekend` | 101.1651 |
| 8 | `day_cos` | 65.6146 |
| 9 | `clouds_all` | 36.3087 |
| 10 | `weather_Clouds` | 12.4848 |

### Interpretation

The SHAP results indicate that **time of day is the dominant driver of predicted traffic volume**.

The three hour-related features (`hour_cos`, `hour`, and `hour_sin`) occupy the top three positions. In particular, `hour_cos` has a substantially larger mean absolute SHAP value than any other predictor. This indicates that the daily cyclical traffic pattern has a strong influence on predicted traffic demand.

Day-related variables (`day_sin`, `day_of_week`, `is_weekend`, and `day_cos`) also contribute meaningfully, indicating that traffic patterns vary according to the day of the week and whether the observation occurs on a weekend.

Weather-related variables have a smaller overall influence. Temperature contributes to the predictions, while cloud coverage and the encoded cloudy-weather indicator have considerably lower mean absolute SHAP values than the main time-related features.

These findings are consistent with the unsupervised analysis in Task 2, where the traffic clusters were differentiated primarily by time of day and traffic volume, with weather severity providing a secondary distinction.

SHAP values describe model behaviour rather than causal effects. Therefore, the results show which variables influence the Random Forest predictions but do not demonstrate that changing those variables would directly cause traffic volume to change.

## 3.13 Deep Learning Model Comparison and Limitations

The neural network achieved:

- MAE = **297.8235 vehicles**
- R-squared = **0.9395**

The comparable Random Forest used for SHAP achieved:

- MAE = **275.08 vehicles**
- R-squared = **0.9436**

The SHAP Random Forest used **30,839 training rows** after the validation split, so its MAE of **275.08** is not identical to the Task 1 Random Forest MAE of **271.21**. Task 1 used the larger training set without a separate neural-network validation split.

Both models therefore produced strong traffic-volume predictions. The Random Forest achieved slightly lower MAE and slightly higher R-squared than the neural network.

The neural network nevertheless demonstrates that a deep learning approach can successfully model traffic demand using the engineered time, weather and holiday-related features.

Training continued for all 50 configured epochs. The best validation loss occurred at epoch **50**, indicating that validation performance was still improving at the end of the configured training period rather than showing a clear early-stopping point.

The final training MSE was **256,941.22**, compared with a final validation MSE of **268,269.97**. The relatively modest difference does not indicate severe overfitting within this experiment.

However, the evaluation uses a random data split. Because traffic observations are sequential, nearby observations may appear in both training and test sets. Performance may therefore be more optimistic than performance on a completely future time period. A production system should also be evaluated using a chronological holdout set.

SHAP was applied to the comparable Random Forest rather than directly to the neural network because tree-based SHAP provides a computationally practical and interpretable explanation of the same traffic-volume prediction problem.

In [12]:
# ------------------------------------------------------------
# Task 3.14 - Final validation
# ------------------------------------------------------------

task3_validation = {
    "Neural network trained":
        nn_model is not None,

    "Neural network predictions created":
        len(nn_predictions) == len(y_test),

    "Neural network MAE calculated":
        np.isfinite(nn_mae),

    "Neural network R-squared calculated":
        np.isfinite(nn_r2),

    "Training history available":
        len(history.history["loss"]) > 0,

    "Validation history available":
        len(history.history["val_loss"]) > 0,

    "Comparable Random Forest trained":
        shap_rf_model is not None,

    "SHAP values calculated":
        shap_values is not None,

    "SHAP feature importance created":
        len(shap_importance) == len(feature_columns),

    "SHAP sample analysed":
        len(shap_sample) == 100
}


logger.info(
    "Task 3 final validation completed."
)

print("TASK 3 FINAL VALIDATION")
print("-" * 60)

for check, passed in task3_validation.items():
    status = "PASS" if passed else "FAIL"
    print(f"{check}: {status}")

all_checks_passed = all(
    task3_validation.values()
)

print("-" * 60)

if all_checks_passed:
    print(
        "TASK 3 STATUS: PASS - "
        "All required checks completed."
    )
else:
    print(
        "TASK 3 STATUS: FAIL - "
        "Review the failed checks."
    )

2026-10-07 22:02:57,706 - INFO - __main__ - Task 3 final validation completed.


TASK 3 FINAL VALIDATION
------------------------------------------------------------
Neural network trained: PASS
Neural network predictions created: PASS
Neural network MAE calculated: PASS
Neural network R-squared calculated: PASS
Training history available: PASS
Validation history available: PASS
Comparable Random Forest trained: PASS
SHAP values calculated: PASS
SHAP feature importance created: PASS
SHAP sample analysed: PASS
------------------------------------------------------------
TASK 3 STATUS: PASS - All required checks completed.


## Task 3 Summary

Task 3 developed a feed-forward neural network for traffic-demand prediction and applied SHAP explainability to a comparable Random Forest model trained on the same prediction problem.

The neural network used 25 engineered time, weather and holiday-related predictors and contained two hidden layers with 64 and 32 neurons. It achieved an MAE of **297.8235 vehicles** and an R-squared value of **0.9395** on the held-out test set.

The comparable Random Forest achieved an MAE of **275.08 vehicles** and an R-squared value of **0.9436**, slightly outperforming the neural network.

SHAP analysis identified time-related variables as the dominant contributors to traffic-volume predictions. `hour_cos` had the highest mean absolute SHAP value, followed by `hour` and `hour_sin`. Day-of-week variables also contributed meaningfully, while weather-related variables generally had smaller effects.

These explainability findings are consistent with Task 2, where time of day was also the strongest differentiator of historical traffic conditions.

The results demonstrate that both neural-network and tree-based approaches can model traffic demand effectively. However, the evaluation uses a random train/test split, so performance may be optimistic relative to prediction on a completely future time period.

SHAP explains the behaviour of the comparable Random Forest model rather than establishing causal relationships between the predictors and traffic volume.